# Uni-modal Integrated PTRS + PRS — Tissue or Cell-Type

For one modality at a time (`MODEL_VERSION ∈ {'tissue', 'ct'}`), build the
**unified PTRS** on that modality's cross-cohort-consistent feature shortlist
(read from `meta_model_exploration_unified.ipynb`'s output), then evaluate 7
integration methods that combine it with variant-level **PRS-CS** and **PRS-CSx**
(altPRS: ϕ=auto/EUR and ϕ=auto/META respectively).

Evaluation: GACRS test (25 % of GACRS train+test with seed 0) + CAMP-only Balanced
(64v64 × 100 bootstrap). External cohort evaluation (CAMP+1KG / CAMP+GTEx) has
been removed.


## 0. Config — set `MODEL_VERSION` and re-run


In [ ]:
# ============================================================
# INPUT_ROOT — upstream PTRS + PRS melt CSVs + pheno files.
#   INPUT_ROOT/combine/data/1/ptrs_results-concat_39_gacrs_train_test/  (tissue PTRS)
#   INPUT_ROOT/combine/data/1/ptrs_results-concat_17CT_gacrs_train_test/ (CT PTRS)
#   INPUT_ROOT/combine/data/1/ptrs_results-concat_39_camp_1k1k/          (tissue CAMP)
#   INPUT_ROOT/combine/data/1/ptrs_results-concat_17CT_camp_gtex/        (CT CAMP)
#   INPUT_ROOT/files/03_prscs-*.csv                                      (PRS melt CSVs)
#
# OUTPUT_ROOT — where results land.
#   OUTPUT_ROOT/meta_model_<MODEL_VERSION>/consistent_features.csv       (INPUT: feature shortlist)
#   OUTPUT_ROOT/integrated_ptrs_prs_<MODEL_VERSION>/                     (OUTPUT: this notebook's results)
# ============================================================
from pathlib import Path
INPUT_ROOT  = Path('/Users/nancyh/Desktop/hartwell/gene_model/score')
OUTPUT_ROOT = Path('/Users/nancyh/Desktop/asthma-prs-study-fresh/09_ptrs-unified_model-evaluation/data/predictions')

# ---- pick which analysis to run ----
MODEL_VERSION = 'tissue'   # 'tissue' or 'ct'  (set by runner)
# ------------------------------------

assert MODEL_VERSION in ('tissue', 'ct'), "MODEL_VERSION must be 'tissue' or 'ct'"

CONFIG = {
    'tissue': {
        'feature_label_plural': 'tissues',
        'feature_count_label':  '39 GTEx tissues (consistent shortlist)',
        'focus_ptrs_base':      str(INPUT_ROOT / 'combine' / 'data' / '1' / 'ptrs_results-concat_39_gacrs_train_test'),
        'camp_ptrs_base':       str(INPUT_ROOT / 'combine' / 'data' / '1' / 'ptrs_results-concat_39_camp_1k1k'),
        'meta_model_dir':       'meta_model_tissue',
        'camp_only_mask_fn':    lambda idx: ~idx.str.match(r'^\d'),  # drop numeric 1KG IDs
    },
    'ct': {
        'feature_label_plural': 'cell_types',
        'feature_count_label':  '17 OneK1K cell types (consistent shortlist)',
        'focus_ptrs_base':      str(INPUT_ROOT / 'combine' / 'data' / '1' / 'ptrs_results-concat_17CT_gacrs_train_test'),
        'camp_ptrs_base':       str(INPUT_ROOT / 'combine' / 'data' / '1' / 'ptrs_results-concat_17CT_camp_gtex'),
        'meta_model_dir':       'meta_model_ct',
        'camp_only_mask_fn':    lambda idx: idx.str.startswith('CA'),  # keep CAMP IDs only
    },
}
cfg = CONFIG[MODEL_VERSION]

# Feature shortlist is READ from the meta_model output — no hardcoded lists.
CONSISTENT_CSV = OUTPUT_ROOT / cfg['meta_model_dir'] / 'consistent_features.csv'
assert CONSISTENT_CSV.exists(), (
    f"Feature shortlist not found: {CONSISTENT_CSV}\n"
    f"Run meta_model_exploration_unified.ipynb (MODEL_VERSION='{MODEL_VERSION}') first."
)

# altPRS configuration (matches prscs_evaluation.ipynb)
PRSCS_PHI    = 'ϕ=auto'
PRSCSX_PHI   = 'ϕ=auto'
PRSCSX_LDREF = 'META'

ARTIFACT_DIR = OUTPUT_ROOT / f'integrated_ptrs_prs_{MODEL_VERSION}'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"INPUT_ROOT    = {INPUT_ROOT}")
print(f"OUTPUT_ROOT   = {OUTPUT_ROOT}")
print(f"MODEL_VERSION = {MODEL_VERSION}  ({cfg['feature_count_label']})")
print(f"Feature list  <- {CONSISTENT_CSV}")
print(f"altPRS        : PRS-CS  ϕ={PRSCS_PHI}, LDREF=EUR")
print(f"                PRS-CSx ϕ={PRSCSX_PHI}, LDREF={PRSCSX_LDREF}")
print(f"Artifacts    -> {ARTIFACT_DIR}")


## 1. Imports & helpers


In [16]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
)
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.base import clone
from sklearn.utils import resample
from scipy.stats import ttest_ind, rankdata
import matplotlib.pyplot as plt
import seaborn as sns


def odds_ratio_quantile(y_true, y_pred, q=0.25):
    thresh_high = np.quantile(y_pred, 1 - q)
    thresh_low = np.quantile(y_pred, q)
    top = y_true[y_pred >= thresh_high]
    bottom = y_true[y_pred <= thresh_low]
    a = top.sum(); b = len(top) - a
    c = bottom.sum(); d = len(bottom) - c
    if b == 0 or c == 0:
        return np.inf
    return (a * d) / (b * c)


def append_predictions(records, sample_ids, scores, y_true, **labels):
    """Append per-sample prediction rows to a long-format list."""
    y_arr = np.asarray(y_true)
    s_arr = np.asarray(scores)
    for sid, s, y in zip(sample_ids, s_arr, y_arr):
        rec = {'sample_id': sid, 'score': float(s), 'y_true': int(y)}
        rec.update(labels)
        records.append(rec)


## 2. Load PTRS on the consistent-features shortlist

Load per-feature PTRS scores for the shortlist read from `consistent_features.csv`
across GACRS (train+test) and CAMP. The CAMP+external file is loaded then
filtered to CAMP-only samples (no external held-out evaluation).


In [ ]:
# Read the shortlist produced by meta_model_exploration_unified.ipynb
focus_tissues = pd.read_csv(CONSISTENT_CSV)['Feature'].tolist()
print(f"Selected {cfg['feature_label_plural']} (consistent shortlist): {len(focus_tissues)}")
for t in focus_tissues:
    print(f"  - {t}")

# GACRS train+test per-feature PTRS
ptrs_results = {}
for feat in focus_tissues:
    fp = Path(cfg['focus_ptrs_base']) / f'{feat}_results.csv'
    if not fp.exists():
        print(f"Missing: {feat}")
        continue
    df = pd.read_csv(fp, index_col=0)
    df['Tissue'] = feat
    ptrs_results[feat] = df

ptrs_df = pd.concat(ptrs_results.values())
ptrs_df = ptrs_df[['Tissue', 'Keep_Vector', 'asthma']].reset_index()

gacrs_wide = ptrs_df.pivot_table(index='Sample_ID', columns='Tissue', values='Keep_Vector')
gacrs_wide['asthma'] = ptrs_df.drop_duplicates('Sample_ID').set_index('Sample_ID')['asthma']
combined_tissues = sorted([c for c in gacrs_wide.columns if c != 'asthma'])
print(f"\nGACRS train+test: {len(gacrs_wide)} samples, {len(combined_tissues)} {cfg['feature_label_plural']}")
print(f"Cases: {int(gacrs_wide['asthma'].sum())}, Controls: {int((gacrs_wide['asthma']==0).sum())}")


In [ ]:
# CAMP+external per-feature PTRS -> filter to CAMP-only immediately
camp_ptrs = {}
for feat in combined_tissues:
    fp = Path(cfg['camp_ptrs_base']) / f'{feat}_results.csv'
    if not fp.exists():
        print(f"Missing: {feat}")
        continue
    df = pd.read_csv(fp, index_col=0)
    df['Tissue'] = feat
    camp_ptrs[feat] = df

camp_ptrs_df = pd.concat(camp_ptrs.values())
camp_ptrs_df = camp_ptrs_df[['Tissue', 'Keep_Vector', 'asthma']].reset_index()

camp_wide = camp_ptrs_df.pivot_table(index='Sample_ID', columns='Tissue', values='Keep_Vector')
camp_wide['asthma'] = camp_ptrs_df.drop_duplicates('Sample_ID').set_index('Sample_ID')['asthma']
camp_wide = camp_wide.dropna(subset=['asthma'])

# CAMP-only: drop external subcohort (1KG for tissue, GTEx for CT)
camp_only_wide = camp_wide[cfg['camp_only_mask_fn'](camp_wide.index)]
print(f"CAMP-only: {len(camp_only_wide)} samples "
      f"(cases: {int(camp_only_wide['asthma'].sum())}, "
      f"controls: {int((camp_only_wide['asthma']==0).sum())})")


## 3. GACRS train / test split + z-score normalize features

Same partition as `meta_model_exploration_unified.ipynb`: stratified 75/25
within GACRS train+test with `random_state=0`. Feature z-scoring uses train
statistics only, applied to both cohorts.


In [ ]:
all_samples = gacrs_wide.index
all_labels  = gacrs_wide['asthma']
train_ids, test_ids = train_test_split(
    all_samples, test_size=0.25, stratify=all_labels, random_state=0,
)
train_id = train_ids.values
test_id  = test_ids.values
print(f"Train: {len(train_id)} ({int(gacrs_wide.loc[train_id, 'asthma'].sum())} cases)")
print(f"Test:  {len(test_id)}  ({int(gacrs_wide.loc[test_id,  'asthma'].sum())} cases)")

# Z-score using train stats only; apply to CAMP-only too.
norm_stats = {}
for col in combined_tissues:
    m = gacrs_wide.loc[train_id, col].mean()
    s = gacrs_wide.loc[train_id, col].std()
    norm_stats[col] = {'mean': m, 'std': s}
    gacrs_wide[col] = (gacrs_wide[col] - m) / s
    if col in camp_only_wide.columns:
        camp_only_wide[col] = (camp_only_wide[col] - m) / s

X_train = gacrs_wide.loc[train_id, combined_tissues]
X_test  = gacrs_wide.loc[test_id,  combined_tissues]
y_train = gacrs_wide.loc[train_id, 'asthma']
y_test  = gacrs_wide.loc[test_id,  'asthma']
X_camp_only = camp_only_wide[combined_tissues]
y_camp_only = camp_only_wide['asthma']
print('Features z-scored')


## 4. Load PRS-CS / PRS-CSx (altPRS: ϕ=auto/EUR and ϕ=auto/META)

PRS values are read from the same melt CSVs used by
`prscs_evaluation.ipynb`, then z-normalized on GACRS train samples only.


In [ ]:
# altPRS PRS-CS  (ϕ=auto, LDREF=EUR) and PRS-CSx (ϕ=auto, LDREF=META)
files_dir = INPUT_ROOT / 'files'
import_prefix = "03_prscs-prscsx-camp-gtex-onek1k-visualization"

prscs_gacrs_full = pd.read_csv(f"{files_dir}/{import_prefix}_prscs-gacrs-only-data-melt-admixture.csv")
prscs_camp_full  = pd.read_csv(f"{files_dir}/{import_prefix}_prscs_camp-1k1k-data-melt-admixture.csv")
prscsx_gacrs_full = pd.read_csv(f"{files_dir}/{import_prefix}_prscsx_gacrs-only-data-melt-admixture.csv")
prscsx_camp_full  = pd.read_csv(f"{files_dir}/{import_prefix}_prscsx_camp-1k1k-data-melt-admixture.csv")

# The CAMP+1KG file is used as the CAMP source for both tissue and CT; filter
# to CAMP-only samples immediately.
prscs_camp_full  = prscs_camp_full[prscs_camp_full['Population']  == 'CAMP']
prscsx_camp_full = prscsx_camp_full[prscsx_camp_full['Population'] == 'CAMP']

phi_col_cs  = 'PRS-CS(ϕ)'
phi_col_csx = [c for c in prscsx_gacrs_full.columns if 'ϕ' in c][0]

prscs_gacrs = (prscs_gacrs_full[prscs_gacrs_full[phi_col_cs] == PRSCS_PHI]
                 [['IID', 'PRS']].rename(columns={'IID': 'Sample_ID', 'PRS': 'PRS_CS'})
                 .set_index('Sample_ID'))
prscs_camp  = (prscs_camp_full[prscs_camp_full[phi_col_cs] == PRSCS_PHI]
                 [['IID', 'PRS']].rename(columns={'IID': 'Sample_ID', 'PRS': 'PRS_CS'})
                 .set_index('Sample_ID'))

prscsx_gacrs = (prscsx_gacrs_full[(prscsx_gacrs_full[phi_col_csx] == PRSCSX_PHI) &
                                   (prscsx_gacrs_full['LDREF'] == PRSCSX_LDREF)]
                  [['IID', 'PRS']].rename(columns={'IID': 'Sample_ID', 'PRS': 'PRS_CSx'})
                  .set_index('Sample_ID'))
prscsx_camp  = (prscsx_camp_full[(prscsx_camp_full[phi_col_csx] == PRSCSX_PHI) &
                                  (prscsx_camp_full['LDREF'] == PRSCSX_LDREF)]
                  [['IID', 'PRS']].rename(columns={'IID': 'Sample_ID', 'PRS': 'PRS_CSx'})
                  .set_index('Sample_ID'))

# Merge PRS-CS + PRS-CSx per cohort
prs_gacrs = prscs_gacrs.merge(prscsx_gacrs, left_index=True, right_index=True, how='inner')
prs_camp  = prscs_camp.merge(prscsx_camp,  left_index=True, right_index=True, how='inner')

# Z-normalize using GACRS train stats only
valid_train_prs = [s for s in train_id if s in prs_gacrs.index]
prs_norm_stats = {}
for col in ['PRS_CS', 'PRS_CSx']:
    m = prs_gacrs.loc[valid_train_prs, col].mean()
    s = prs_gacrs.loc[valid_train_prs, col].std()
    prs_norm_stats[col] = {'mean': m, 'std': s}
    prs_gacrs[col + '_z'] = (prs_gacrs[col] - m) / s
    prs_camp[col + '_z']  = (prs_camp[col]  - m) / s

print(f"PRS-CS  GACRS/CAMP-only: {len(prscs_gacrs)}/{len(prscs_camp)}")
print(f"PRS-CSx GACRS/CAMP-only: {len(prscsx_gacrs)}/{len(prscsx_camp)}")
print(f"Merged  GACRS/CAMP-only: {len(prs_gacrs)}/{len(prs_camp)}")
print(f"Training samples with PRS: {len(valid_train_prs)}")
print(f"\nPRS-CS  train mean/std: {prs_norm_stats['PRS_CS']['mean']:.3f} / {prs_norm_stats['PRS_CS']['std']:.3f}")
print(f"PRS-CSx train mean/std: {prs_norm_stats['PRS_CSx']['mean']:.3f} / {prs_norm_stats['PRS_CSx']['std']:.3f}")


## 5. Build unified PTRS via RF GridSearch + 5-fold OOF + save predictions

1. Fit `GridSearchCV` on full train → best RF estimator
2. Use the best estimator with `cross_val_predict` (5-fold stratified) for **out-of-fold** train predictions (avoids leakage when used as a feature downstream)
3. For test / external / CAMP-only: use full best-estimator predictions


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={
        'n_estimators': [50, 100, 200],
        'max_depth': [2, 3, 5, None],
        'min_samples_leaf': [10, 20, 50],
        'max_features': [1, 'sqrt', 'log2'],
    },
    cv=skf, scoring='roc_auc', n_jobs=-1, refit=True,
)
rf_grid.fit(X_train, y_train)
unified_model = rf_grid.best_estimator_
print(f"Best RF params: {rf_grid.best_params_}")

unified_train_oof = cross_val_predict(unified_model, X_train, y_train, cv=skf, method='predict_proba')[:, 1]
unified_test      = unified_model.predict_proba(X_test)[:, 1]
unified_camp_only = unified_model.predict_proba(X_camp_only)[:, 1]

print('Unified PTRS computed (RF GridSearch — best estimator):')
print(f"  Train OOF: {len(unified_train_oof)} samples (mean={unified_train_oof.mean():.3f})")
print(f"  Test: {len(unified_test)}")
print(f"  CAMP-only: {len(unified_camp_only)}")
print(f"\nUnified PTRS only — GACRS test AUC: {roc_auc_score(y_test, unified_test):.4f}")
print(f"Unified PTRS only — CAMP-only AUC: {roc_auc_score(y_camp_only, unified_camp_only):.4f}")

# Save unified PTRS predictions
unified_rows = []
append_predictions(unified_rows, X_train.index.tolist(),     unified_train_oof, y_train.values,
                   cohort='GACRS_train_OOF', model_version=MODEL_VERSION)
append_predictions(unified_rows, X_test.index.tolist(),      unified_test,      y_test.values,
                   cohort='GACRS_test',      model_version=MODEL_VERSION)
append_predictions(unified_rows, X_camp_only.index.tolist(), unified_camp_only, y_camp_only.values,
                   cohort='CAMP_only',       model_version=MODEL_VERSION)
unified_predictions_df = pd.DataFrame(unified_rows).rename(columns={'score': 'unified_PTRS'})
unified_predictions_df.to_csv(ARTIFACT_DIR / 'unified_ptrs.csv', index=False)
print(f"\nSaved -> {ARTIFACT_DIR / 'unified_ptrs.csv'}  ({len(unified_predictions_df)} rows)")


## 6. Build 2-feature integrated dataframes (`[unified_PTRS, PRS_z]`)


In [ ]:
# Attach unified PTRS back to each cohort's index
gacrs_wide['unified_PTRS'] = np.nan
gacrs_wide.loc[train_id, 'unified_PTRS'] = unified_train_oof
gacrs_wide.loc[test_id,  'unified_PTRS'] = unified_test
camp_only_wide['unified_PTRS'] = unified_camp_only

df_integrated_gacrs = gacrs_wide[['unified_PTRS', 'asthma'] + combined_tissues].merge(
    prs_gacrs[['PRS_CS_z', 'PRS_CSx_z']], left_index=True, right_index=True, how='inner',
)
df_integrated_camp_only = camp_only_wide[['unified_PTRS', 'asthma'] + combined_tissues].merge(
    prs_camp[['PRS_CS_z', 'PRS_CSx_z']], left_index=True, right_index=True, how='inner',
)

valid_train_int = [s for s in train_id if s in df_integrated_gacrs.index]
valid_test_int  = [s for s in test_id  if s in df_integrated_gacrs.index]

print(f"Integrated GACRS train: {len(valid_train_int)}")
print(f"Integrated GACRS test:  {len(valid_test_int)}")
print(f"Integrated CAMP-only:   {len(df_integrated_camp_only)}")


## 7. Define integration models (7 methods)

In [ ]:
# Integration strategies — match the paper's Methods description exactly.
# 5-fold stratified CV on GACRS-train, shuffle=True, random_state=42.
# Tree/ensemble tuners use AUC; LogisticRegressionCV uses sklearn default score.
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

skf_int = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

integration_models = {
    # 1. Logistic Regression — L2 (ridge), C=1.0 (both are sklearn defaults; stated explicitly for clarity)
    'Logistic Regression': LogisticRegression(
        penalty='l2', C=1.0, solver='liblinear', max_iter=1000,
    ),

    # 2. Elastic Net — fixed l1_ratio=0.5, C selected by 5-fold CV over sklearn default log-scale grid (Cs=10)
    'Elastic Net (CV)': LogisticRegressionCV(
        penalty='elasticnet', solver='saga',
        l1_ratios=[0.5],   # fixed 50/50 L1/L2
        Cs=10,             # sklearn default: 10 log-spaced C values 1e-4..1e4
        cv=5, max_iter=2000, random_state=42,
    ),

    # 3. Random Forest — grid-searched with max_features axis
    'Random Forest (tuned)': GridSearchCV(
        RandomForestClassifier(random_state=42),
        param_grid={
            'n_estimators':     [50, 100, 200],
            'max_depth':        [2, 3, 5, None],
            'min_samples_leaf': [10, 20, 50],
            'max_features':     [1, 'sqrt', 'log2'],
        },
        cv=skf_int, scoring='roc_auc', n_jobs=-1, refit=True,
    ),

    # 4. Gradient Boosting — smaller grid per paper
    'Gradient Boosting (tuned)': GridSearchCV(
        GradientBoostingClassifier(random_state=42),
        param_grid={
            'n_estimators':  [50, 100],
            'max_depth':     [2, 3],
            'learning_rate': [0.05, 0.1],
        },
        cv=skf_int, scoring='roc_auc', n_jobs=-1, refit=True,
    ),

    # 5. Linear SVM (LinearSVC C=0.1) with Platt scaling via CalibratedClassifierCV (method='sigmoid', cv=3)
    'SVM (linear)': CalibratedClassifierCV(
        LinearSVC(C=0.1, dual=False, random_state=42, max_iter=5000),
        method='sigmoid', cv=3,
    ),

    # 6. Stacking-Fixed — LR + RF + GB base learners, LR meta-learner, base cv=5
    'Stacking (LR+RF+GB)': StackingClassifier(
        estimators=[
            ('lr', LogisticRegression(C=0.1, solver='liblinear', max_iter=1000)),
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=3, min_samples_leaf=20, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.05, random_state=42)),
        ],
        final_estimator=LogisticRegression(solver='liblinear', max_iter=1000),
        cv=5,
    ),
    # 7. Rank-Add handled separately below (training-free)
}
print(f"Integration methods: {list(integration_models.keys()) + ['Rank Addition']}")


## 8. Evaluation framework — GACRS test + CAMP-only Full + CAMP-only Balanced


In [ ]:
def evaluate_predictions(name, eval_set, y_true, preds, results, n_repeats=100):
    """Append metric row.
    - 'CAMP-only Balanced': 64v64 × 100 bootstrap.
    - Everything else ('GACRS Test', 'CAMP-only Full'): single-pass metrics + Welch t-test P.
    """
    if eval_set == 'CAMP-only Balanced':
        idx_cases    = np.where(y_true == 1)[0]
        idx_controls = np.where(y_true == 0)[0]
        n_ctrl = len(idx_controls)
        aucs, ors, diffs = [], [], []
        for seed in range(n_repeats):
            rng = np.random.RandomState(seed)
            cs  = rng.choice(idx_cases, size=n_ctrl, replace=False)
            idx = np.concatenate([cs, idx_controls])
            y_b = y_true[idx]; p_b = preds[idx]
            aucs.append(roc_auc_score(y_b, p_b))
            ors.append(odds_ratio_quantile(y_b, p_b))
            g1 = p_b[y_b == 0]; g2 = p_b[y_b == 1]
            diffs.append(g2.mean() - g1.mean())
        results.append({
            'Method': name, 'Eval_Set': eval_set,
            'AUC': float(np.mean(aucs)), 'AUC_std': float(np.std(aucs)),
            'OR':  float(np.mean(ors)),  'OR_std':  float(np.std(ors)),
            'Mean_Diff': float(np.mean(diffs)),
            'AUPRC': np.nan, 'P_Value': np.nan,
            'N_cases': n_ctrl, 'N_controls': n_ctrl,
        })
    else:
        auc   = roc_auc_score(y_true, preds)
        auprc = average_precision_score(y_true, preds)
        OR    = odds_ratio_quantile(y_true, preds)
        g1 = preds[y_true == 0]; g2 = preds[y_true == 1]
        _, p = ttest_ind(g1, g2, equal_var=False)
        results.append({
            'Method': name, 'Eval_Set': eval_set,
            'AUC': auc, 'AUC_std': np.nan,
            'OR':  OR,  'OR_std':  np.nan,
            'Mean_Diff': g2.mean() - g1.mean(),
            'AUPRC': auprc, 'P_Value': p,
            'N_cases': int(y_true.sum()), 'N_controls': int((y_true == 0).sum()),
        })


EVAL_SETS = ['GACRS Test', 'CAMP-only Full', 'CAMP-only Balanced']
print(f"Evaluation sets: {EVAL_SETS}")


## 9. Integrated evaluation: `[unified_PTRS, PRS_z]` for both PRS-CS and PRS-CSx

In [ ]:
# Build all 2-feature inputs up front
def make_2feature(prs_col):
    return (
        df_integrated_gacrs.loc[valid_train_int, ['unified_PTRS', prs_col]],
        df_integrated_gacrs.loc[valid_test_int,  ['unified_PTRS', prs_col]],
        df_integrated_camp_only[['unified_PTRS', prs_col]],
    )


y_int_train     = df_integrated_gacrs.loc[valid_train_int, 'asthma']
y_int_test      = df_integrated_gacrs.loc[valid_test_int,  'asthma']
y_int_camp_only = df_integrated_camp_only['asthma']

results_prscs  = []
results_prscsx = []
integrated_predictions = []


def run_integration(prs_col, prs_label, results_target):
    Xtr, Xte, Xco = make_2feature(prs_col)
    print(f"\n--- Integrated [unified_PTRS, {prs_col}] --- train={len(Xtr)} test={len(Xte)} camp_only={len(Xco)}")

    for method_name, model_template in integration_models.items():
        model = clone(model_template)
        model.fit(Xtr, y_int_train)

        test_preds = model.predict_proba(Xte)[:, 1]
        co_preds   = model.predict_proba(Xco)[:, 1]

        evaluate_predictions(method_name, 'GACRS Test',         y_int_test.values,      test_preds, results_target)
        evaluate_predictions(method_name, 'CAMP-only Full',     y_int_camp_only.values, co_preds,   results_target)
        evaluate_predictions(method_name, 'CAMP-only Balanced', y_int_camp_only.values, co_preds,   results_target)

        append_predictions(integrated_predictions, Xte.index.tolist(), test_preds, y_int_test.values,
                           method=method_name, prs_type=prs_label, eval_set='GACRS Test',
                           model_version=MODEL_VERSION)
        append_predictions(integrated_predictions, Xco.index.tolist(), co_preds, y_int_camp_only.values,
                           method=method_name, prs_type=prs_label, eval_set='CAMP-only',
                           model_version=MODEL_VERSION)
        print(f"  Done: {method_name}")

    # Rank Addition — no training, equal-weight rank sum
    def rank_norm(s):
        return rankdata(s) / len(s)
    rank_test = rank_norm(Xte['unified_PTRS']) + rank_norm(Xte[prs_col])
    rank_co   = rank_norm(Xco['unified_PTRS']) + rank_norm(Xco[prs_col])
    evaluate_predictions('Rank Addition', 'GACRS Test',         y_int_test.values,      rank_test, results_target)
    evaluate_predictions('Rank Addition', 'CAMP-only Full',     y_int_camp_only.values, rank_co,   results_target)
    evaluate_predictions('Rank Addition', 'CAMP-only Balanced', y_int_camp_only.values, rank_co,   results_target)
    append_predictions(integrated_predictions, Xte.index.tolist(), rank_test, y_int_test.values,
                       method='Rank Addition', prs_type=prs_label, eval_set='GACRS Test',
                       model_version=MODEL_VERSION)
    append_predictions(integrated_predictions, Xco.index.tolist(), rank_co, y_int_camp_only.values,
                       method='Rank Addition', prs_type=prs_label, eval_set='CAMP-only',
                       model_version=MODEL_VERSION)
    print(f"  Done: Rank Addition")


run_integration('PRS_CS_z',  'PRS-CS',  results_prscs)
run_integration('PRS_CSx_z', 'PRS-CSx', results_prscsx)

results_prscs_df  = pd.DataFrame(results_prscs)
results_prscsx_df = pd.DataFrame(results_prscsx)
print('\n=== Integrated PRS-CS  ===')
print(results_prscs_df[['Method','Eval_Set','AUC','AUC_std','OR','OR_std']].to_string(index=False))
print('\n=== Integrated PRS-CSx ===')
print(results_prscsx_df[['Method','Eval_Set','AUC','AUC_std','OR','OR_std']].to_string(index=False))


## 10. Baseline: direct integration `[all_features + PRS_z]` with the same 7 methods

In [ ]:
results_direct = []
direct_predictions = []

for prs_label, prs_col in [('PRS-CS', 'PRS_CS_z'), ('PRS-CSx', 'PRS_CSx_z')]:
    features = combined_tissues + [prs_col]
    Xtr = df_integrated_gacrs.loc[valid_train_int, features]
    Xte = df_integrated_gacrs.loc[valid_test_int,  features]
    Xco = df_integrated_camp_only[features]
    print(f"\n--- Direct baseline {prs_label} ({len(features)} features) ---")

    for method_name, model_template in integration_models.items():
        model = clone(model_template)
        model.fit(Xtr, y_int_train)
        test_preds = model.predict_proba(Xte)[:, 1]
        co_preds   = model.predict_proba(Xco)[:, 1]
        full_name  = f"Direct ({prs_label}) + {method_name}"
        evaluate_predictions(full_name, 'GACRS Test',         y_int_test.values,      test_preds, results_direct)
        evaluate_predictions(full_name, 'CAMP-only Full',     y_int_camp_only.values, co_preds,   results_direct)
        evaluate_predictions(full_name, 'CAMP-only Balanced', y_int_camp_only.values, co_preds,   results_direct)
        append_predictions(direct_predictions, Xte.index.tolist(), test_preds, y_int_test.values,
                           method=method_name, prs_type=prs_label, eval_set='GACRS Test',
                           model_version=MODEL_VERSION)
        append_predictions(direct_predictions, Xco.index.tolist(), co_preds, y_int_camp_only.values,
                           method=method_name, prs_type=prs_label, eval_set='CAMP-only',
                           model_version=MODEL_VERSION)
        print(f"  Done: {method_name}")

results_direct_df = pd.DataFrame(results_direct)
print('\n=== Direct (all features + PRS) baseline — top 10 ===')
print(results_direct_df.head(10).to_string(index=False))


## 11. Summary + save outputs

In [ ]:
all_results = pd.concat([
    results_prscs_df.assign(Approach='Unified PTRS + PRS-CS'),
    results_prscsx_df.assign(Approach='Unified PTRS + PRS-CSx'),
    results_direct_df.assign(Approach='Direct'),
], ignore_index=True)


def show_eval(eval_set):
    print(f"\n=== {eval_set} ===")
    sub = all_results[all_results['Eval_Set'] == eval_set].sort_values('AUC', ascending=False)
    print(sub[['Approach', 'Method', 'AUC', 'AUC_std', 'OR', 'OR_std', 'Mean_Diff', 'P_Value']].head(10).to_string(index=False))


for es in EVAL_SETS:
    show_eval(es)

print('\n=== Best method per evaluation set ===')
for es in EVAL_SETS:
    sub  = all_results[all_results['Eval_Set'] == es]
    best = sub.loc[sub['AUC'].idxmax()]
    if es == 'CAMP-only Balanced':
        print(f"{es}: {best['Approach']} | {best['Method']} — AUC={best['AUC']:.4f} ± {best['AUC_std']:.4f}, OR={best['OR']:.2f}")
    else:
        print(f"{es}: {best['Approach']} | {best['Method']} — AUC={best['AUC']:.4f}, OR={best['OR']:.2f}, P={best['P_Value']:.4f}")


In [ ]:
# Persist final outputs
all_results.to_csv(ARTIFACT_DIR / 'all_results.csv', index=False)
pd.DataFrame(integrated_predictions).to_csv(ARTIFACT_DIR / 'integrated_predictions.csv', index=False)
pd.DataFrame(direct_predictions).to_csv(ARTIFACT_DIR / 'direct_predictions.csv',       index=False)

print(f"Saved -> {ARTIFACT_DIR / 'all_results.csv'}")
print(f"Saved -> {ARTIFACT_DIR / 'integrated_predictions.csv'}  ({len(integrated_predictions)} rows)")
print(f"Saved -> {ARTIFACT_DIR / 'direct_predictions.csv'}      ({len(direct_predictions)} rows)")
